# More voice features + a t-SNE map of the speakers

1. Cut **4-second solo snippets** for every speaker (`make_snippets.py`).
2. Measure **features** for each snippet:
   - **openSMILE eGeMAPS**: 88 standard voice features computed from the
     audio (pitch, loudness, voice quality, formants, rhythm).
   - **Transcript features**: speed and pauses from the word timestamps.
3. Draw a **t-SNE map**: every snippet is a dot, and snippets that sound
   alike end up close together.

Install once in the terminal: `uv add opensmile scikit-learn`

## 1. Setup

In [ ]:
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import opensmile
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm

from make_snippets import (LABEL_DIR, RAW_DIR, SAMPLE_RATE, load_snippet,
                           make_snippets, read_json)

OUT_DIR = Path("../data")          # results are cached here
SNIPPET_SEC = 4
SNIPPETS_PER_SPEAKER = 10
TEST_SPEAKERS = None               # None = everyone, e.g. 35 for a quick test
SEED = 42                          # same snippets and same map every run

GENDER_COLORS = {"male": "#2a6fdb", "female": "#d6457a"}

## 2. Speakers and snippets

In [ ]:
def load_tracks():
    """One row per speaker track: who is speaking, in which conversation."""
    rows = []
    for split in ("train", "val", "test"):
        conversations = sorted(
            p for p in (RAW_DIR / split).iterdir() if p.is_dir())
        for conv in conversations:
            for track in ("a", "b"):
                meta = read_json(
                    LABEL_DIR / conv.name / f"speaker_{track}_meta.json")
                rows.append({
                    "speaker_id": meta["speaker_id"],
                    "split": split,
                    "conversation_id": conv.name,
                    "track": track,
                    "gender": meta["gender"],
                    "age": meta["age_range"],
                })
    return pd.DataFrame(rows)


tracks = load_tracks()
if TEST_SPEAKERS:
    all_ids = sorted(tracks.speaker_id.unique())
    keep = all_ids[::len(all_ids) // TEST_SPEAKERS][:TEST_SPEAKERS]
    tracks = tracks[tracks.speaker_id.isin(keep)]

speakers = (tracks.groupby("speaker_id")[["gender", "age"]]
            .first().reset_index())
print(f"{len(tracks)} speaker tracks, {len(speakers)} speakers")

In [ ]:
snippets, rejected = make_snippets(tracks, snippet_sec=SNIPPET_SEC,
                                   per_speaker=SNIPPETS_PER_SPEAKER,
                                   seed=SEED)
print(f"{len(snippets)} snippets from {snippets.speaker_id.nunique()} "
      f"speakers")
print("windows rejected by the audio checks:", dict(rejected))

## 3. Measure the features

**openSMILE eGeMAPS** gives 88 numbers per snippet. The most useful ones,
in plain words:

| Group | openSMILE feature | Meaning |
|---|---|---|
| Pitch | `F0semitoneFrom27.5Hz_sma3nz_amean` | how high the voice is |
| Tone / intonation | `F0semitoneFrom27.5Hz_sma3nz_stddevNorm` | how much the pitch moves |
| Loudness | `loudness_sma3_amean` | how loud (partly the microphone) |
| Rhythm | `loudnessPeaksPerSec` | loudness peaks ≈ syllables per second |
| Rhythm | `MeanVoicedSegmentLengthSec` | length of voiced stretches |
| Pauses | `MeanUnvoicedSegmentLength` | length of gaps between them |
| Voice quality | `jitterLocal…`, `shimmerLocaldB…`, `HNRdBACF…` | rough, breathy or clear voice |
| Vocal tract | `F1frequency…`, `F2…`, `F3…` | formants: shape of mouth and throat |
| Voice colour | `mfcc1…` … `mfcc4…`, `slope…`, `hammarbergIndex…` | timbre |

**Transcript features** (more exact than audio estimates for speed):
`words_per_sec`, `word_ms`, `speech_share`, `pauses_per_10s`, `pause_ms`.

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)


def voice_features(audio):
    """88 eGeMAPS features of one snippet (openSMILE)."""
    return smile.process_signal(audio, SAMPLE_RATE).iloc[0].to_dict()


def transcript_features(snippet):
    """Speed and pauses of one snippet, from the word timestamps."""
    path = (LABEL_DIR / snippet.conversation_id
            / f"speaker_{snippet.track}_transcript.json")
    start, end = snippet.start_sec, snippet.end_sec
    words = [w for w in read_json(path)["words"]
             if w["end"] > start and w["start"] < end]
    # words cut by the snippet edge only count for the part inside it
    talk_sec = sum(min(w["end"], end) - max(w["start"], start)
                   for w in words)
    gaps = [b["start"] - a["end"] for a, b in zip(words, words[1:])]
    pauses = [gap for gap in gaps if gap > 0.25]
    length = end - start
    return {
        "words_per_sec": len(words) / talk_sec if talk_sec else np.nan,
        "word_ms": 1000 * talk_sec / len(words) if words else np.nan,
        "speech_share": talk_sec / length,
        "pauses_per_10s": 10 * len(pauses) / length,
        "pause_ms": 1000 * np.mean(pauses) if pauses else 0.0,
    }


def measure_features(snippet):
    """All features of one snippet: openSMILE voice + transcript."""
    return {**voice_features(load_snippet(snippet)),
            **transcript_features(snippet)}

In [ ]:
csv_name = (f"tsne_features_test{TEST_SPEAKERS}.csv" if TEST_SPEAKERS
            else "tsne_features.csv")
features_csv = OUT_DIR / csv_name

if features_csv.exists():
    feats = pd.read_csv(features_csv)
    print("loaded", features_csv)
else:
    rows = [measure_features(s)
            for s in tqdm(snippets.itertuples(), total=len(snippets),
                          desc="features")]
    feats = pd.concat([snippets.reset_index(drop=True), pd.DataFrame(rows)],
                      axis=1)
    feats.to_csv(features_csv, index=False)
    print("saved", features_csv)

feats = feats.merge(speakers, on="speaker_id")
INFO_COLUMNS = ["speaker_id", "split", "conversation_id", "track",
                "start_sec", "end_sec", "gender", "age"]
FEATURE_COLUMNS = [c for c in feats.columns if c not in INFO_COLUMNS]
print(f"{len(feats)} snippets x {len(FEATURE_COLUMNS)} features")

## 4. t-SNE map

Each snippet has 93 numbers. **t-SNE** turns them into a 2D map:

1. In the 93-dimensional feature space it finds, for every snippet,
   which other snippets are its **close neighbours** (sound most alike).
2. It places all snippets on a 2D map and **moves them step by step**
   until neighbours in the feature space are also neighbours on the map.
3. The horizontal position is **t-SNE dimension 1**, the vertical
   position **t-SNE dimension 2**.

So the two dimensions are **not features** and have no unit. To make the
maps readable, every axis label says **which features change most along
that axis** (Spearman correlation `r`; +1 = always rises, 0 = unrelated).

- Features are standardised first (mean 0, std 1), so no feature
  dominates just because of its units.
- `random_state` fixes the result, so the map is the same every run.
- Only **neighbourhoods** are reliable: "these dots are close" is
  meaningful, "this cluster is twice as far" is not.

In [ ]:
def standardise(df, columns):
    """Fill gaps with the median and scale every column to mean 0, std 1."""
    values = df[columns].fillna(df[columns].median())
    values = values.loc[:, values.std() > 0]           # drop constant columns
    return ((values - values.mean()) / values.std()).to_numpy()


def run_tsne(values, perplexity=30):
    """2D t-SNE coordinates (fixed seed)."""
    tsne = TSNE(n_components=2, perplexity=perplexity, init="pca",
                random_state=SEED)
    return tsne.fit_transform(values)


# Plain-language names for the openSMILE / transcript features.
# Labels read as "low <name>" on one side and "high <name>" on the other.
PLAIN_NAMES = [
    ("F0semitone", "pitch"),
    ("loudness", "loudness"),
    ("equivalentSoundLevel", "loudness"),
    ("HNR", "voice clarity"),
    ("jitter", "jitter (roughness)"),
    ("shimmer", "shimmer (roughness)"),
    ("logRelF0", "breathiness"),
    ("F1", "F1 (vowel openness)"),
    ("F2", "F2 (vowel frontness)"),
    ("F3", "F3 formant"),
    ("hammarberg", "darkness (Hammarberg)"),
    ("alphaRatio", "brightness (alpha ratio)"),
    ("slope", "spectral slope"),
    ("spectralFlux", "spectral change"),
    ("mfcc", "MFCC voice colour"),
    ("VoicedSegmentsPerSec", "syllable rate"),
    ("loudnessPeaksPerSec", "syllable rate"),
    ("MeanVoicedSegment", "voiced-stretch length"),
    ("StddevVoicedSegment", "rhythm variability"),
    ("MeanUnvoicedSegment", "gap length"),
    ("StddevUnvoicedSegment", "gap variability"),
    ("words_per_sec", "speaking rate"),
    ("word_ms", "word length"),
    ("speech_share", "talking share"),
    ("pauses_per_10s", "pause rate"),
    ("pause_ms", "pause length"),
]


def plain_name(feature):
    """Plain-language name of a feature."""
    for prefix, name in PLAIN_NAMES:
        if feature.startswith(prefix):
            return name
    return feature


def describe_axis(position, df, columns, top=2):
    """Describe one map axis by the features that correlate most with it.

    Returns a two-part label, e.g.
    '<- low pitch, low loudness  |  high pitch, high loudness ->  (r = 0.71)'.
    """
    corr = df[columns].corrwith(pd.Series(position, index=df.index),
                                method="spearman").dropna()
    best = {}                                  # strongest feature per name
    for feature, r in corr.items():
        name = plain_name(feature)
        if name not in best or abs(r) > abs(best[name]):
            best[name] = r
    strongest = sorted(best.items(), key=lambda item: -abs(item[1]))[:top]
    low = ", ".join(("low " if r > 0 else "high ") + name
                    for name, r in strongest)
    high = ", ".join(("high " if r > 0 else "low ") + name
                     for name, r in strongest)
    return f"← {low}   |   {high} →   (r = {abs(strongest[0][1]):.2f})"


def label_axes(ax, x, y, df, columns, compact=False):
    """Name both axes after the features that change most along them."""
    top = 1 if compact else 2                  # small panels: one feature
    dim1 = describe_axis(x, df, columns, top=top)
    dim2 = describe_axis(y, df, columns, top=top)
    size = 7 if compact else 9
    ax.set_xlabel(f"t-SNE dimension 1\n{dim1}", fontsize=size)
    ax.set_ylabel(f"t-SNE dimension 2\n{dim2}", fontsize=size)
    ax.tick_params(labelsize=7)
    ax.grid(color="#eeeeee", linewidth=0.5)
    ax.set_axisbelow(True)


def add_caption(fig, how_to_read, what_it_shows):
    """Write a 'How to read' / 'What it shows' box under a figure."""
    width = int(fig.get_figwidth() * 14)       # characters per line
    text = (textwrap.fill("HOW TO READ: " + how_to_read, width) + "\n\n"
            + textwrap.fill("WHAT IT SHOWS: " + what_it_shows, width))
    fig.text(0.01, -0.01, text, ha="left", va="top", fontsize=10,
             bbox={"boxstyle": "round", "facecolor": "#f7f7f7",
                   "edgecolor": "#cccccc"})


def same_gender_share(x, y, genders):
    """Share of dots whose nearest dot on the map has the same gender."""
    points = np.column_stack([x, y])
    _, index = NearestNeighbors(n_neighbors=2).fit(points).kneighbors(points)
    genders = np.asarray(genders)
    return float((genders[index[:, 1]] == genders).mean())


def label_group_centre(ax, x, y, text, color):
    """Write a label at the centre of a group of dots."""
    ax.text(np.median(x), np.median(y), text, color="white", fontsize=10,
            ha="center", va="center", fontweight="bold",
            bbox={"boxstyle": "round", "facecolor": color, "alpha": 0.85})


coords = run_tsne(standardise(feats, FEATURE_COLUMNS))
feats["tsne_x"], feats["tsne_y"] = coords[:, 0], coords[:, 1]

### 4a. Male and female voices

Each dot is one 4-second snippet. Dots close together sound alike.

In [ ]:
gender_share = same_gender_share(feats.tsne_x, feats.tsne_y, feats.gender)
pitch_col = "F0semitoneFrom27.5Hz_sma3nz_amean"
median_pitch_hz = (27.5 * 2 ** (feats.groupby("gender")[pitch_col].median()
                                / 12)).round()

fig, ax = plt.subplots(figsize=(11, 8.5))
for gender, color in GENDER_COLORS.items():
    d = feats[feats.gender == gender]
    ax.scatter(d.tsne_x, d.tsne_y, s=6, color=color, alpha=0.7,
               label=f"{gender}: {len(d)} snippets, "
                     f"{d.speaker_id.nunique()} speakers")
for gender, color in GENDER_COLORS.items():
    d = feats[feats.gender == gender]
    label_group_centre(ax, d.tsne_x, d.tsne_y,
                       f"mostly {gender} voices\n"
                       f"median pitch {median_pitch_hz[gender]:.0f} Hz",
                       color)
label_axes(ax, feats.tsne_x, feats.tsne_y, feats, FEATURE_COLUMNS)
ax.set_title("All 4-second snippets on one map, coloured by gender")
ax.legend(frameon=False, markerscale=3, loc="upper left")
add_caption(
    fig,
    "each dot is one 4 s snippet of one speaker, placed using all 93 "
    "features. Dots that are close sound alike. The axis labels say which "
    "features change most from one side of the map to the other.",
    f"male and female voices fill different halves of the map, driven "
    f"mostly by pitch and loudness (left to right). "
    f"{100 * gender_share:.0f}% of snippets have a nearest neighbour of "
    f"the same gender. The mixed zone in the middle holds low female and "
    f"high male voices.")
plt.tight_layout()
plt.show()

### 4b. What organises the map?

The same map, coloured by six single features.

In [ ]:
COLOR_FEATURES = {
    "F0semitoneFrom27.5Hz_sma3nz_amean":
        ("pitch", "semitones above 27.5 Hz"),
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm":
        ("pitch variation (intonation)", "monotone → expressive"),
    "words_per_sec": ("speaking rate", "words per second"),
    "loudness_sma3_amean": ("loudness", "quiet → loud"),
    "HNRdBACF_sma3nz_amean":
        ("voice clarity (HNR)", "dB: breathy/rough → clear"),
    "mfcc1_sma3_amean": ("voice colour (MFCC 1)", "spectral shape"),
}


def map_strength(column):
    """How strongly one feature follows the map (max |r| with an axis)."""
    r = max(abs(feats[column].corr(feats.tsne_x, method="spearman")),
            abs(feats[column].corr(feats.tsne_y, method="spearman")))
    if r >= 0.5:
        return r, "strong gradient: shapes the map"
    if r >= 0.25:
        return r, "partial gradient"
    return r, "speckled: barely shapes the map"


fig, axes = plt.subplots(2, 3, figsize=(18, 11.5))
strengths = {}
for ax, (column, (title, unit)) in zip(axes.flat, COLOR_FEATURES.items()):
    low, high = feats[column].quantile([0.02, 0.98])   # ignore extremes
    points = ax.scatter(feats.tsne_x, feats.tsne_y, s=5, c=feats[column],
                        cmap="viridis", vmin=low, vmax=high)
    fig.colorbar(points, ax=ax, shrink=0.8, label=unit)
    label_axes(ax, feats.tsne_x, feats.tsne_y, feats, FEATURE_COLUMNS,
               compact=True)
    r, verdict = map_strength(column)
    strengths[title] = r
    ax.set_title(f"{title}\n{verdict} (r = {r:.2f})", fontsize=12)
fig.suptitle("The same t-SNE map, coloured by single features "
             "(yellow = high, purple = low)", fontsize=14)
ranked = sorted(strengths, key=strengths.get, reverse=True)
add_caption(
    fig,
    "every panel shows the same dots as 4a. Only the colour changes: it is "
    "the value of one feature. A smooth colour change across the map means "
    "that feature decides where snippets land; a random speckle means it "
    "does not. r = how strongly the feature follows the map (0 to 1).",
    f"the map is organised mainly by {ranked[0]} and {ranked[1]}; "
    f"{ranked[-1]} and {ranked[-2]} hardly matter for where a snippet "
    f"lands.")
plt.tight_layout()
plt.show()

### 4c. Do a speaker's snippets stay together?

12 example speakers, each snippet linked to the speaker's centre.

In [ ]:
def speaker_spread(df):
    """Average distance of each speaker's snippets from their centre."""
    centre = df.groupby("speaker_id")[["tsne_x", "tsne_y"]].transform("mean")
    distance = np.hypot(df.tsne_x - centre.tsne_x, df.tsne_y - centre.tsne_y)
    return distance.groupby(df.speaker_id).mean()


spread = speaker_spread(feats)
all_ids = sorted(feats.speaker_id.unique())
examples = all_ids[::max(1, len(all_ids) // 12)][:12]

fig, ax = plt.subplots(figsize=(12, 9.5))
ax.scatter(feats.tsne_x, feats.tsne_y, s=4, color="#e3e3e3",
           label="all other snippets")
for i, speaker_id in enumerate(examples):
    d = feats[feats.speaker_id == speaker_id]
    color = plt.cm.tab20(i)
    cx, cy = d.tsne_x.mean(), d.tsne_y.mean()
    for x, y in zip(d.tsne_x, d.tsne_y):
        ax.plot([cx, x], [cy, y], color=color, linewidth=0.8, alpha=0.7)
    ax.scatter(d.tsne_x, d.tsne_y, s=22, color=color, edgecolor="black",
               linewidth=0.3,
               label=f"{speaker_id} ({d.gender.iloc[0]}), "
                     f"spread {spread[speaker_id]:.0f}")
    ax.text(cx, cy, str(i + 1), fontsize=9, fontweight="bold", ha="center",
            va="center", color="white",
            bbox={"boxstyle": "circle", "facecolor": color})
label_axes(ax, feats.tsne_x, feats.tsne_y, feats, FEATURE_COLUMNS)
ax.set_title("12 example speakers: each snippet linked to the speaker's "
             "centre (numbered circle)")
ax.legend(fontsize=7, frameon=False, loc="center left",
          bbox_to_anchor=(1, 0.5), title="speaker (gender), spread")
tightest = spread[examples].idxmin()
loosest = spread[examples].idxmax()
add_caption(
    fig,
    "grey = all snippets. Each colour is one speaker; lines join each of "
    "their snippets to their centre (numbered circle). Spread = average "
    "line length. Short lines = the features recognise this voice "
    "consistently; long lines = the snippets look like different people.",
    f"voices differ a lot in how consistent they look. The tightest example "
    f"is {tightest} (spread {spread[tightest]:.0f}), the loosest {loosest} "
    f"(spread {spread[loosest]:.0f}). Across all speakers the median spread "
    f"is {spread.median():.0f}, so most voices are only partly recognised "
    f"from 4 s of audio.")
plt.tight_layout()
plt.show()

## 5. How well do the features tell speakers apart?

For every snippet we find its **nearest neighbour** among all other
snippets (in the full feature space, not the 2D map) and check whether it
is the **same speaker**.

- **any conversation**: the neighbour may come from the same recording.
- **other conversations only**: the neighbour must come from a *different*
  recording. This is the fair test, because the same microphone and room
  can make snippets look alike. Only speakers with 2+ conversations count.

In [ ]:
FEATURE_GROUPS = {
    "all features": FEATURE_COLUMNS,
    "timbre (MFCC & spectrum)": [
        c for c in FEATURE_COLUMNS
        if c.startswith(("mfcc", "slope", "alpha", "hammarberg",
                         "spectralFlux"))],
    "voice quality": [c for c in FEATURE_COLUMNS
                      if c.startswith(("jitter", "shimmer", "HNR",
                                       "logRelF0"))],
    "formants": [c for c in FEATURE_COLUMNS
                 if c.startswith(("F1", "F2", "F3"))],
    "loudness": [c for c in FEATURE_COLUMNS
                 if c.startswith(("loudness_", "equivalentSoundLevel"))],
    "pitch & intonation": [c for c in FEATURE_COLUMNS if c.startswith("F0")],
    "speed & pauses (transcript)": ["words_per_sec", "word_ms",
                                    "speech_share", "pauses_per_10s",
                                    "pause_ms"],
    "rhythm (audio)": [
        "loudnessPeaksPerSec", "VoicedSegmentsPerSec",
        "MeanVoicedSegmentLengthSec", "StddevVoicedSegmentLengthSec",
        "MeanUnvoicedSegmentLength", "StddevUnvoicedSegmentLength"],
}


def same_speaker_rate(df, columns, other_conversation=False):
    """Share of snippets whose nearest neighbour is the same speaker."""
    values = standardise(df, columns)
    speaker = df.speaker_id.to_numpy()
    conversation = df.conversation_id.to_numpy()
    n_conversations = df.groupby("speaker_id").conversation_id.transform(
        "nunique").to_numpy()

    _, index = NearestNeighbors(n_neighbors=50).fit(values).kneighbors(values)
    hits = []
    for i, neighbours in enumerate(index):
        neighbours = neighbours[neighbours != i]
        if other_conversation:
            if n_conversations[i] < 2:
                continue
            neighbours = neighbours[conversation[neighbours]
                                    != conversation[i]]
        if len(neighbours):
            hits.append(speaker[neighbours[0]] == speaker[i])
    return float(np.mean(hits))


scores = pd.DataFrame({
    "any conversation": {
        name: same_speaker_rate(feats, cols)
        for name, cols in FEATURE_GROUPS.items()},
    "other conversations only": {
        name: same_speaker_rate(feats, cols, other_conversation=True)
        for name, cols in FEATURE_GROUPS.items()},
})
chance = 1 / feats.speaker_id.nunique()

In [ ]:
fair = scores["other conversations only"]
order = fair.sort_values().index
y = np.arange(len(order))

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(y + 0.2, 100 * scores.loc[order, "any conversation"], height=0.4,
        color="#9fd3b8", label="neighbour from any conversation")
ax.barh(y - 0.2, 100 * fair[order], height=0.4, color="#2e9e6b",
        label="neighbour from another conversation (fair test)")
for yi, group in zip(y, order):
    times = fair[group] / chance
    ax.text(100 * fair[group] + 0.5, yi - 0.2,
            f"{100 * fair[group]:.0f}%  ({times:.0f}× chance)",
            va="center", fontsize=8)
ax.axvline(100 * chance, color="k", linestyle="--", linewidth=1,
           label=f"random guess ({100 * chance:.1f}%)")
ax.set_yticks(y)
ax.set_yticklabels([f"{g} ({len(FEATURE_GROUPS[g])} features)"
                    for g in order])
ax.set_xlabel("% of snippets whose most similar other snippet "
              "is the same speaker")
ax.set_ylabel("feature group used")
ax.set_title("Which features recognise a speaker?\n"
             f"{feats.speaker_id.nunique()} speakers, "
             f"{len(feats)} snippets of 4 s")
ax.grid(axis="x", color="#eeeeee")
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right", fontsize=8)
best = fair.drop("all features").idxmax()
gap = 100 * (scores.loc["all features", "any conversation"]
             - fair["all features"])
add_caption(
    fig,
    "for each snippet we look up the most similar other snippet (using "
    "only that feature group) and check if it is the same person. Longer "
    "bar = the group recognises speakers better. Dark bars only allow "
    "matches from a different recording, which is the fair test.",
    f"{best} does almost all the work ({100 * fair[best]:.0f}% on its own, "
    f"{100 * fair['all features']:.0f}% with all features, about "
    f"{fair['all features'] / chance:.0f}× better than guessing). "
    f"Pitch, speed and rhythm barely identify anyone. The light bars are "
    f"about {gap:.0f} points higher than the dark ones: part of the "
    f"matching is the same microphone and room, not the person.")
plt.tight_layout()
plt.show()
(100 * scores).round(1).sort_values("other conversations only",
                                    ascending=False)

### 5b. One map per feature group

t-SNE using **only** one group of features at a time, coloured by gender.

The streaks in the speed & pauses map are not a bug: those features take
only a few distinct values in 4 s (e.g. 0, 2.5 or 5 pauses per 10 s), so
many snippets are identical and t-SNE lines them up.

In [ ]:
MAP_GROUPS = ["timbre (MFCC & spectrum)", "voice quality", "formants",
              "pitch & intonation", "speed & pauses (transcript)",
              "rhythm (audio)"]

fig, axes = plt.subplots(2, 3, figsize=(19, 12.5))
gender_split = {}
for ax, group in zip(axes.flat, MAP_GROUPS):
    columns = FEATURE_GROUPS[group]
    group_coords = run_tsne(standardise(feats, columns))
    for gender, color in GENDER_COLORS.items():
        is_gender = (feats.gender == gender).to_numpy()
        ax.scatter(group_coords[is_gender, 0], group_coords[is_gender, 1],
                   s=4, color=color, alpha=0.6, label=gender)
    label_axes(ax, group_coords[:, 0], group_coords[:, 1], feats, columns,
               compact=True)
    gender_split[group] = same_gender_share(
        group_coords[:, 0], group_coords[:, 1], feats.gender)
    ax.set_title(f"{group} ({len(columns)} features)\n"
                 f"same speaker: {100 * fair[group]:.0f}%   |   "
                 f"neighbour same gender: {100 * gender_split[group]:.0f}%",
                 fontsize=11)
axes.flat[0].legend(frameon=False, markerscale=3)
fig.suptitle("t-SNE with one feature group at a time, coloured by gender",
             fontsize=14)
best_gender = max(gender_split, key=gender_split.get)
add_caption(
    fig,
    "each panel is a separate map built from one feature group only. "
    "'same speaker' = how often that group finds the same person (from "
    "chart 5); 'neighbour same gender' = how often a dot's nearest dot has "
    "the same gender (50% = no separation, 100% = perfect).",
    f"{best_gender} separates men and women best "
    f"({100 * gender_split[best_gender]:.0f}% same-gender neighbours) but "
    f"tells individuals apart much less well than timbre. Pitch tells you "
    f"'male or female', timbre tells you 'who'. Speed and rhythm form no "
    f"useful structure in 4 s.")
plt.tight_layout()
plt.show()

## 6. One dot per speaker

Each speaker's snippets are averaged, and t-SNE is run on the averages.

In [ ]:
speaker_means = (feats.groupby(["speaker_id", "gender"])[FEATURE_COLUMNS]
                 .mean().reset_index())
speaker_coords = run_tsne(standardise(speaker_means, FEATURE_COLUMNS),
                          perplexity=15)
speaker_means["tsne_x"] = speaker_coords[:, 0]
speaker_means["tsne_y"] = speaker_coords[:, 1]

# speakers whose closest other speaker has the other gender
points = speaker_coords
_, nearest = NearestNeighbors(n_neighbors=2).fit(points).kneighbors(points)
genders = speaker_means.gender.to_numpy()
crossover = genders[nearest[:, 1]] != genders
speaker_gender_share = 1 - crossover.mean()

fig, (ax_gender, ax_pitch) = plt.subplots(1, 2, figsize=(19, 8.5))
for gender, d in speaker_means.groupby("gender"):
    ax_gender.scatter(d.tsne_x, d.tsne_y, s=45, color=GENDER_COLORS[gender],
                      alpha=0.8, edgecolor="white", linewidth=0.5,
                      label=f"{gender} ({len(d)} speakers)")
ax_gender.scatter(speaker_means.tsne_x[crossover],
                  speaker_means.tsne_y[crossover], s=110,
                  facecolor="none", edgecolor="black", linewidth=1,
                  label=f"closest voice is the other gender "
                        f"({crossover.sum()})")
for gender, color in GENDER_COLORS.items():
    d = speaker_means[speaker_means.gender == gender]
    label_group_centre(ax_gender, d.tsne_x, d.tsne_y, f"{gender} voices",
                       color)
ax_gender.legend(frameon=False, loc="upper left")
ax_gender.set_title("Speakers, coloured by gender")

points_pitch = ax_pitch.scatter(speaker_means.tsne_x, speaker_means.tsne_y,
                                s=45, c=speaker_means[pitch_col],
                                cmap="viridis", edgecolor="white",
                                linewidth=0.5)
fig.colorbar(points_pitch, ax=ax_pitch,
             label="average pitch (semitones above 27.5 Hz)")
ax_pitch.set_title("Speakers, coloured by average pitch")

for ax in (ax_gender, ax_pitch):
    label_axes(ax, speaker_means.tsne_x, speaker_means.tsne_y,
               speaker_means, FEATURE_COLUMNS)
fig.suptitle(f"t-SNE of speakers: each dot = one person "
             f"(average of their snippets), n={len(speaker_means)}",
             fontsize=14)
add_caption(
    fig,
    "each dot is one person: the average of all their snippets. People "
    "close together have similar voices overall. Left: gender; black "
    "circles mark people whose closest voice belongs to the other gender. "
    "Right: the same map coloured by average pitch.",
    f"men and women fill different halves of the map: "
    f"{100 * speaker_gender_share:.0f}% of speakers have a same-gender "
    f"closest voice (snippets in 4a: {100 * gender_share:.0f}%). Pitch "
    f"changes smoothly across the map. The {crossover.sum()} circled "
    f"speakers sit among voices of the other gender (e.g. women with low "
    f"voices); they are the ones a model is most likely to confuse.")
plt.tight_layout()
plt.show()